# Ethnicity subgroups across frozen prediction models


In [ ]:
from pathlib import Path
for required_source in ("score106.py", "metrics.py"):
    required_path = Path.cwd().parent / "sources" / required_source
    if not required_path.is_file():
        raise FileNotFoundError(f"Required analysis module not found: {required_path}")

from lifelines.exceptions import ConvergenceWarning
import warnings

REVISION_RUNS = ("reviewer_full249_score_5fold", "reviewer_raw106_score")
for run_id in REVISION_RUNS:
    from pathlib import Path
    import os, json, copy, hashlib, shutil, ast
    PACKAGE = Path.cwd().parent
    assert ((Path.cwd() / '00_shared_config.ipynb')).exists(), 'The shared configuration notebook must be in the working directory.'
    PROJECT = Path(os.environ.get("CHD_OUTPUT_ROOT", "../outputs")).resolve()
    BASE_RUN = run_id
    NEW_RUN = 'reviewer_raw106_score'
    COHORT = 'prs_raw106_complete'

    BOOTSTRAPS = 1000
    os.environ.update(CHD_OUTPUT_ROOT=str(PROJECT), CHD_RUN_ID=BASE_RUN, CHD_COHORT_ID=COHORT,
                      CHD_SENSITIVITY_OUTPUT_DIR=str(PROJECT/'runs'/BASE_RUN),
                      CHD_SHARED_NOTEBOOK=str((Path.cwd() / '00_shared_config.ipynb')), CHD_BOOTSTRAP_N=str(BOOTSTRAPS))
    get_ipython().run_line_magic('run', '"'+str((Path.cwd() / '00_shared_config.ipynb'))+'"')
    df_train,df_val,df_test,final_feature_sets,base_manifest=validate_pipeline_artifacts(True)
    BASE = PROJECT/'runs'/BASE_RUN
    frames={'train':df_train,'val':df_val,'test':df_test}
    print('Verified existing partitions:', {k:len(v) for k,v in frames.items()})

    ETHNICITY_CSV = os.environ.get('CHD_ETHNICITY_CSV') or None
    WHITE={1,1001,1002,1003}
    NONWHITE={2,3,4,5,6,2001,2002,2003,2004,3001,3002,3003,3004,4001,4002,4003}
    paths=[Path(ETHNICITY_CSV)] if ETHNICITY_CSV else [Path(e['path']) for e in base_manifest['input_files']]
    parts=[]
    for path in paths:
        h=pd.read_csv(path,nrows=0).columns.tolist()
        cols=[c for c in h if c in ['21000','21000-0.0','21000_0_0','p21000_i0','p21000_i0_a0']]
        if not cols: continue
        if len(cols)!=1: raise RuntimeError(f'Expected one baseline ethnicity column: {path}: {cols}')
        eid=find_eid_col(h)
        d=pd.read_csv(path,usecols=[eid,cols[0]],low_memory=False).rename(columns={eid:'Eid',cols[0]:'Ethnicity_Code'})
        if d.Eid.duplicated().any(): raise RuntimeError('Duplicate ethnicity EIDs')
        parts.append(d.set_index('Eid').Ethnicity_Code)
    if not parts: raise FileNotFoundError('Field 21000 was not found. Set CHD_ETHNICITY_CSV to a file containing Eid and baseline Field 21000.')
    s=parts[0]
    for p in parts[1:]:
        overlap=s.index.intersection(p.index)
        if not s.loc[overlap].fillna(-999).equals(p.loc[overlap].fillna(-999)): raise RuntimeError('Conflicting ethnicity sources')
        s=s.combine_first(p)
    counts=[]
    for split,f in frames.items():
        if not f.Eid.isin(s.index).all(): raise RuntimeError('The ethnicity data do not cover every cohort EID.')
        f['Ethnicity_Code']=pd.to_numeric(f.Eid.map(s),errors='coerce')
        unknown_codes=set(f.Ethnicity_Code.dropna().unique())-WHITE-NONWHITE-{-1,-3}
        if unknown_codes: raise RuntimeError(f'Unexpected ethnicity codes: {unknown_codes}')
        f['Ethnicity_Group']=np.select([f.Ethnicity_Code.isin(WHITE),f.Ethnicity_Code.isin(NONWHITE)],['White','Known_nonWhite_mixed_other'],default='Unclassified')
        for group,g in f.groupby('Ethnicity_Group'):
            counts.append(dict(Split=split,Group=group,N=len(g),Events=int(g.Event.sum()),
                               Events_10y=int(((g.Event==1)&(g.Time<=10)).sum()),Observed_beyond_10y=int((g.Time>10).sum())))
    OUT=PROJECT/'runs'/BASE_RUN/'04_results'/'reviewer_ethnicity_all_prediction_models'
    OUT.mkdir(exist_ok=True,parents=True)
    pd.DataFrame(counts).to_csv(OUT/'ethnicity_split_counts.csv',index=False)
    allf=pd.concat(list(frames.values()),ignore_index=True)
    observed=allf.Ethnicity_Group.value_counts().to_dict()
    print('Observed overall:',observed)
    expected={'White':195305,'Known_nonWhite_mixed_other':9412,'Unclassified':875}
    if COHORT=='prs_raw106_complete' and observed!=expected:
        raise RuntimeError('Ethnicity counts differ from the locked analysis population.')
    allf.groupby(['Ethnicity_Group','Ethnicity_Code'],dropna=False).size().rename('N').to_csv(OUT/'ethnicity_detailed_composition.csv')
    # Audit the genetic-grouping field that already shaped the eligible cohort.
    gparts=[]
    for path in [Path(e['path']) for e in base_manifest['input_files']]:
        h=pd.read_csv(path,nrows=0).columns.tolist(); eid=find_eid_col(h)
        gc=[c for c in h if (parse_ukb_col(c) is not None and parse_ukb_col(c)[0]==22006
                             and parse_ukb_col(c)[1:]==(0,0))]
        if not gc: continue
        d=pd.read_csv(path,usecols=[eid,gc[0]],low_memory=False).rename(columns={eid:'Eid',gc[0]:'Genetic_Ethnic_Code'})
        if d.Eid.duplicated().any(): raise RuntimeError('Duplicate genetic-ethnicity EIDs')
        gparts.append(d.set_index('Eid').Genetic_Ethnic_Code)
    if gparts:
        gs=gparts[0]
        for p in gparts[1:]:
            overlap=gs.index.intersection(p.index)
            if not gs.loc[overlap].fillna(-999).equals(p.loc[overlap].fillna(-999)):
                raise RuntimeError('Conflicting Field22006 sources')
            gs=gs.combine_first(p)
        allf['Genetic_Ethnic_Code']=pd.to_numeric(allf.Eid.map(gs),errors='coerce')
        pd.crosstab([allf.Ethnicity_Group,allf.Ethnicity_Code],allf.Genetic_Ethnic_Code,
                    dropna=False,margins=True).to_csv(OUT/'selfreported_by_genetic_ethnicity_22006.csv')
    else:
        print('Field 22006 not found in raw files; cohort filtering rule:',base_manifest.get('cohort_switches',{}))

    import importlib.util
    METRICS_SOURCE = PACKAGE / "sources" / "metrics.py"
    if not METRICS_SOURCE.is_file():
        raise FileNotFoundError(f"Required analysis module not found: {METRICS_SOURCE}")
    spec=importlib.util.spec_from_file_location('review_metrics',METRICS_SOURCE)
    M=importlib.util.module_from_spec(spec); spec.loader.exec_module(M)

    # Parallel bootstrap: preserve serial RNG draws, metric functions and result order.
    from joblib import Parallel, delayed, parallel_backend
    from threadpoolctl import threadpool_limits
    import sys
    try:
        from joblib.externals import cloudpickle
    except ImportError:
        import cloudpickle
    sys.modules["review_metrics"]=M
    if hasattr(cloudpickle,"register_pickle_by_value"):
        cloudpickle.register_pickle_by_value(M)
    N_JOBS=int(os.environ.get("CHD_ETHNICITY_N_JOBS","24"))
    if N_JOBS<1: raise ValueError("CHD_ETHNICITY_N_JOBS must be positive")
    BLOCK_SIZE=10
    print(f"Ethnicity bootstrap: {N_JOBS} processes; one internal thread per process",flush=True)

    def _index_blocks(n, reps, seed):
        rng=np.random.default_rng(seed)
        for start in range(0,int(reps),BLOCK_SIZE):
            # Same per-replicate call as the serial implementation.
            yield np.stack([rng.integers(0,n,size=n)
                            for _ in range(min(BLOCK_SIZE,int(reps)-start))])

    def _dispatch(worker,n,reps,seed,*args):
        with parallel_backend("loky",inner_max_num_threads=1):
            return Parallel(n_jobs=min(N_JOBS,int(reps)),batch_size=1,
                            pre_dispatch=N_JOBS,verbose=10)(
                delayed(worker)(block,*args) for block in _index_blocks(n,reps,seed))

    def _abs_block(block,ref,test,risks,h):
        result={k:[] for k in risks}
        with threadpool_limits(limits=1):
            for ix in block:
                tb=test.iloc[ix].reset_index(drop=True)
                for k,risk in risks.items():
                    try: result[k].append(M.uno_c(ref,tb,risk[ix],h))
                    except (ValueError,ZeroDivisionError): pass
        return result

    def parallel_absolute(ref,test,risks,h,reps,seed):
        parts=_dispatch(_abs_block,len(test),reps,seed,
            ref[["Time","Event"]],test[["Time","Event"]],risks,h)
        return {k:[v for p in parts for v in p[k]] for k in risks}

    def _uno_block(block,ref,test,rnew,rref,h):
        values=[]
        with threadpool_limits(limits=1):
            for ix in block:
                tb=test.iloc[ix].reset_index(drop=True)
                if tb.Event.nunique()<2: continue
                try:
                    values.append(M.uno_c(ref,tb,rnew[ix],h)-M.uno_c(ref,tb,rref[ix],h))
                except Exception: continue
        return values

    def parallel_delta_uno(ref,test,rnew,rref,h,bootstrap_n,seed=M.BOOTSTRAP_SEED):
        rnew=np.asarray(rnew,float); rref=np.asarray(rref,float)
        new=M.uno_c(ref,test,rnew,h); old=M.uno_c(ref,test,rref,h)
        parts=_dispatch(_uno_block,len(test),bootstrap_n,seed,
            ref[["Time","Event"]],test[["Time","Event"]],rnew,rref,h)
        values=[v for p in parts for v in p]
        lo,hi=M.quantile_ci(values)
        return dict(Uno_C_New=new,Uno_C_Reference=old,Delta_Uno_C=new-old,
            Delta_Uno_C_Lower95=lo,Delta_Uno_C_Upper95=hi,Bootstrap_N_Used=len(values))

    def _nri_block(block,test,pref,pnew,h,cutpoints):
        values={k:[] for k in ["NRI_Categorical","NRI_Continuous","IDI"]}
        with threadpool_limits(limits=1):
            for ix in block:
                tb=test.iloc[ix].reset_index(drop=True)
                try:
                    result=M.ipcw_nri_idi_once(tb,pref[ix],pnew[ix],h,cutpoints=cutpoints)
                    for k in values: values[k].append(result[k])
                except Exception: continue
        return values

    def parallel_nri(test,pref,pnew,h,bootstrap_n,seed=M.BOOTSTRAP_SEED):
        pref=np.asarray(pref,float); pnew=np.asarray(pnew,float)
        result=M.ipcw_nri_idi_once(test,pref,pnew,h,cutpoints=M.NRI_CUTPOINTS)
        parts=_dispatch(_nri_block,len(test),bootstrap_n,seed,
            test[["Time","Event"]],pref,pnew,h,M.NRI_CUTPOINTS)
        values={k:[v for p in parts for v in p[k]]
                for k in ["NRI_Categorical","NRI_Continuous","IDI"]}
        for k,v in values.items():
            lo,hi=M.quantile_ci(v); result[k+"_Lower95"]=lo; result[k+"_Upper95"]=hi
        result["Bootstrap_N_Used"]=min(len(v) for v in values.values())
        return result

    # Evaluate saved models only. All model registration checks remain active.
    reps=['Clinical_plus_PRS_plus_NMR_score','Clinical_plus_PRS_plus_SpearmanNMR','Clinical_plus_PRS_plus_Raw106NMR']
    core=['Clinical_only','Clinical_plus_PRS','Clinical_plus_NMR_score','Clinical_plus_PRS_plus_NMR_score']
    specs=list(dict.fromkeys([(f,'Cox PH') for f in core]+[(f,a) for f in reps for a in ['Cox PH','Elastic-net Cox','XGBoost Cox','DeepSurv']]))
    records={}; registry=[]
    for f,a in specs:
        if not set(f'22009_PC{i}' for i in range(1,11)).issubset(final_feature_sets[f]):
            raise RuntimeError('Missing PC1-10: '+f)
        r=load_all_risk_records({f:final_feature_sets[f]},[a])[(MAIN_SCENARIO_NAME,f,a)]
        k=f+'__'+a; records[k]=r
        registry.append(dict(Key=k,Feature=f,Algorithm=a,Run=BASE_RUN))
    pd.DataFrame(registry).to_csv(OUT/'model_registry.csv',index=False)
    features=list(records)
    df_train,df_val,df_test=frames['train'],frames['val'],frames['test']

    def _fit_absolute_calibrator(train_frame, train_risk, label):
        train_risk = np.asarray(train_risk, dtype=float).reshape(-1)
        center = float(np.mean(train_risk))
        scale = float(np.std(train_risk, ddof=0))
        if not np.isfinite(scale) or scale <= 0:
            raise ValueError(f"Degenerate training risk for {label}")

        calibration_frame = pd.DataFrame({
            "Time": train_frame["Time"].astype(float).to_numpy(),
            "Event": train_frame["Event"].astype(int).to_numpy(),
            "Risk_Score": (train_risk - center) / scale,
        })
        model = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(
                calibration_frame,
                duration_col="Time",
                event_col="Event",
                show_progress=False,
                fit_options={"max_steps": 2000, "step_size": 0.5},
            )
        return {
            "label": label,
            "model": model,
            "center": center,
            "scale": scale,
            "method": "train-fitted unpenalized Cox recalibration; extended convergence settings",
        }

    cals = {
        k: _fit_absolute_calibrator(df_train, r["train_risk"], k)
        for k, r in records.items()
    }

    # Same model comparisons in every ethnic subgroup, specified before inspecting results.
    pairs=[]
    base='Clinical_plus_PRS__Cox PH'
    for f in reps: pairs.append((f+'__Cox PH',base,'Representation_vs_Clinical_PRS'))
    pairs.extend([(core[1]+'__Cox PH',core[0]+'__Cox PH','Core_increment'),(core[2]+'__Cox PH',core[0]+'__Cox PH','Core_increment'),(core[3]+'__Cox PH',core[0]+'__Cox PH','Core_increment'),(core[3]+'__Cox PH',core[2]+'__Cox PH','Core_increment')])
    for f in reps:
        for a in ['Elastic-net Cox','XGBoost Cox','DeepSurv']:
            pairs.append((f+'__'+a,f+'__Cox PH','Algorithm_vs_Cox'))
    for a in ['Cox PH','Elastic-net Cox']:
        for ref in reps[1:]: pairs.append((reps[0]+'__'+a,ref+'__'+a,'Score_vs_representation'))
    rows=[]; diffs=[]; skips=[]
    for gi,group in enumerate(['White','Known_nonWhite_mixed_other','Unclassified']):
        idx=np.flatnonzero(df_test.Ethnicity_Group.eq(group)); ti=np.flatnonzero(df_train.Ethnicity_Group.eq(group))
        test=df_test.iloc[idx].reset_index(drop=True); reference=df_train.iloc[ti].reset_index(drop=True)
        for h in [5.,10.]:
            if group=='Unclassified':
                skips.append(dict(Group=group,Horizon=h,Reason='counts only; unknown ethnicity')); continue
            try: M.horizon_support_audit(reference,test,[h])
            except Exception as e:
                skips.append(dict(Group=group,Horizon=h,Reason=str(e))); continue
            print('START',BASE_RUN,group,h,'models=',len(features),flush=True)
            risks={k:np.asarray(records[k]['test_risk'],float)[idx] for k in features}
            preds={k:M.predict_absolute_risk(cals[k],risks[k],h) for k in features}
            ibs={k:M.subject_level_ipcw_ibs(reference,test,cals[k],risks[k],h) for k in features}
            boot=parallel_absolute(reference,test,risks,h,BOOTSTRAPS,2026+gi+int(h))
            for k in features:
                lo,hi=M.quantile_ci(boot[k]) if len(boot[k])>=.9*BOOTSTRAPS else (np.nan,np.nan)
                rows.append(dict(Run=BASE_RUN,Group=group,Horizon=h,Model=k,N=len(test),Events_total=int(test.Event.sum()),Events_by_horizon=int(((test.Event==1)&(test.Time<=h)).sum()),Uno_C=M.uno_c(reference,test,risks[k],h),Lower95=lo,Upper95=hi,Bootstrap_valid=len(boot[k]),IBS=ibs[k]['IBS'],Mean_predicted=preds[k].mean(),Observed_KM=M.km_observed_risk(test,h)))
                M.calibration_by_quantile(test,preds[k],h,groups=5).to_csv(OUT/f'{group}_{int(h)}y_{k}_calibration.csv',index=False)
            # Reuse matched draws when all absolute bootstrap replicates are valid.
            for new,old,kind in pairs:
                if len(boot[new])==BOOTSTRAPS and len(boot[old])==BOOTSTRAPS:
                    v=np.asarray(boot[new])-np.asarray(boot[old]); lo,hi=M.quantile_ci(v)
                    delta=dict(Delta_Uno_C=M.uno_c(reference,test,risks[new],h)-M.uno_c(reference,test,risks[old],h),Delta_Uno_C_Lower95=lo,Delta_Uno_C_Upper95=hi,Bootstrap_N_Used=len(v))
                else:
                    delta=parallel_delta_uno(reference,test,risks[new],risks[old],h,bootstrap_n=BOOTSTRAPS)
                di=M.paired_bootstrap_delta_ibs(ibs[new],ibs[old],bootstrap_n=BOOTSTRAPS)
                # Reclassification for Cox representation additions; algorithm comparison uses C and IBS.
                results=[('Uno',delta),('IBS',di)]
                if kind=='Representation_vs_Clinical_PRS':
                    results.append(('NRI_IDI',parallel_nri(test,preds[old],preds[new],h,bootstrap_n=BOOTSTRAPS)))
                row=dict(Run=BASE_RUN,Group=group,Horizon=h,New=new,Reference=old,Comparison=kind)
                for prefix,result in results:
                    for key,value in result.items(): row[prefix+'_bootstrap_valid' if key=='Bootstrap_N_Used' else key]=value
                diffs.append(row)
            M.survival_dca(test,preds,h).to_csv(OUT/f'{group}_{int(h)}y_dca.csv',index=False)
            pd.DataFrame(rows).to_csv(OUT/'all_model_performance.csv',index=False)
            pd.DataFrame(diffs).to_csv(OUT/'all_paired_differences.csv',index=False)
    pd.DataFrame(skips).to_csv(OUT/'support_and_skips.csv',index=False)
    atomic_write_json(dict(status='completed_with_skips' if skips else 'completed',run=BASE_RUN,score=base_manifest['nmr_score'],bootstrap_n=BOOTSTRAPS,workers=N_JOBS,design='post-review exploratory internal self-reported ethnicity subgroup assessment',scope='Result2 core Cox; Result3 three representations; Result4 four algorithms; calibration and DCA for these frozen predictions',not_included=['Full249 direct 249-input sensitivity model','joint nine-stratum Cox and KM','continuous score interaction','individual metabolite associations/interactions'],uncertainty='conditional on frozen models',multiplicity='pointwise 95% confidence intervals; exploratory comparisons; no multiplicity adjustment',bootstrap_note='paired Uno intervals reuse common absolute-model bootstrap draws; seed differs from earlier standalone delta intervals',ethnicity_note='Inherited Field22006 rule retains eligible value and missing values; not external ancestry validation'),OUT/'analysis_manifest.json')
    print('DONE:',OUT,flush=True)
